# FinBERT Inference on Recovered URL Titles (v10 Pipeline)

**Purpose**: v9 의 FinBERT 분석 대상이 45K (0.38%) 에 머물러 있던 한계를, URL 슬러그 복구로 회수한 영어 후보 3-4M rows 에 대해 ProsusAI/finbert 를 돌려 v10 버전을 생성한다.

## 입력
`finbert_input_candidates.parquet` — VM 에서 사전 추출한 파일.
컬럼: `url, slug_text, first_date_raw, source_domain, slug_quality, in_v9`

## 파이프라인
1. fastText `lid.176.ftz` 로 slug_text 언어 재검증 → 영어 + confidence ≥ 0.5 만 통과
2. ProsusAI/finbert (BERT-base, 110M) GPU 배치 추론
3. 출력: `finbert_v2.parquet` (url, finbert_neg, finbert_neu, finbert_pos, finbert_score, finbert_z, lid_en_conf, lid_lang)

## 예상 비용 / 시간 (T4 또는 A100 기준, 추론)
- langid (fastText) : ~5 min for 6M rows
- FinBERT : T4 기준 ~200 titles/s → 3.5M rows ≈ 5 hours. A100 ≈ 1.5 hours.

## 사용 절차
Runtime > Change runtime type > GPU (T4/A100)

## 1. 의존성 설치

In [ ]:
!pip install -q transformers==4.44.0 accelerate pyarrow pandas tqdm fasttext-wheel

## 2. 입력 파일 업로드

VM 에서 받은 `finbert_input_candidates.parquet` 를 업로드.
대안: Google Drive 마운트 후 경로 수정.

In [ ]:
from google.colab import files, drive
# Option A: 직접 업로드
# uploaded = files.upload()
# INPUT_PARQ = list(uploaded.keys())[0]

# Option B (권장): Drive 마운트
drive.mount('/content/drive')
INPUT_PARQ = '/content/drive/MyDrive/nabi/finbert_input_candidates.parquet'
OUTPUT_PARQ = '/content/drive/MyDrive/nabi/finbert_v2.parquet'

## 3. fastText lid.176 다운로드 + langid

In [ ]:
import os, urllib.request
MODEL_PATH = '/content/lid.176.ftz'
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz',
        MODEL_PATH)

import fasttext
LID = fasttext.load_model(MODEL_PATH)

In [ ]:
import pandas as pd, pyarrow.parquet as pq, pyarrow as pa
import numpy as np
from tqdm.auto import tqdm

pf = pq.ParquetFile(INPUT_PARQ)
print('rows =', pf.metadata.num_rows, 'row_groups =', pf.num_row_groups)

In [ ]:
# langid : 전체를 벡터로 예측
def lid_predict(texts):
    # fasttext predict 는 list 지원
    cleaned = [t.replace('\n', ' ').replace('\r', ' ') for t in texts]
    labels, probs = LID.predict(cleaned, k=1)
    langs = [l[0].replace('__label__', '') for l in labels]
    confs = [float(p[0]) for p in probs]
    return langs, confs

## 4. FinBERT 모델 로드

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = 'ProsusAI/finbert'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
mdl.eval()

# ProsusAI/finbert label mapping: 0=positive, 1=negative, 2=neutral
LABEL_MAP = {0: 'positive', 1: 'negative', 2: 'neutral'}

## 5. 배치 추론 루프 (row-group 단위 스트리밍)

In [ ]:
import time
BATCH = 256  # T4 16GB OK. OOM 나면 128 로 낮출 것.
LID_THRESH = 0.5  # 영어 confidence 하한
MAX_LEN = 64      # slug_text 길이 통계상 95%가 15단어 이하 → token 60 이내

writer = None
t0 = time.time()
total_scored = 0
total_rows = 0

for rg in range(pf.num_row_groups):
    tbl = pf.read_row_group(rg)
    df = tbl.to_pandas()
    del tbl

    # langid
    langs, confs = lid_predict(df['slug_text'].fillna('').astype(str).tolist())
    df['lid_lang'] = langs
    df['lid_en_conf'] = confs
    # 영어 필터
    en_mask = (df['lid_lang'] == 'en') & (df['lid_en_conf'] >= LID_THRESH)
    df_en = df[en_mask].reset_index(drop=True)
    total_rows += len(df)

    if len(df_en) == 0:
        print(f'rg {rg+1}/{pf.num_row_groups}: no english rows')
        continue

    # FinBERT inference
    texts = df_en['slug_text'].tolist()
    negs = np.zeros(len(texts), dtype=np.float32)
    neus = np.zeros(len(texts), dtype=np.float32)
    poss = np.zeros(len(texts), dtype=np.float32)

    for i in tqdm(range(0, len(texts), BATCH),
                  desc=f'rg{rg+1}/{pf.num_row_groups}'):
        chunk = texts[i:i+BATCH]
        enc = tok(chunk, padding=True, truncation=True,
                  max_length=MAX_LEN, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = mdl(**enc).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        # order: 0=positive,1=negative,2=neutral
        poss[i:i+len(chunk)] = probs[:, 0]
        negs[i:i+len(chunk)] = probs[:, 1]
        neus[i:i+len(chunk)] = probs[:, 2]

    df_en['finbert_pos'] = poss
    df_en['finbert_neg'] = negs
    df_en['finbert_neu'] = neus
    df_en['finbert_score'] = poss - negs   # -1..+1
    df_en['finbert_model'] = 'ProsusAI/finbert'
    cols_out = ['url', 'slug_text', 'first_date_raw', 'source_domain',
                'slug_quality', 'in_v9', 'lid_lang', 'lid_en_conf',
                'finbert_pos', 'finbert_neg', 'finbert_neu', 'finbert_score', 'finbert_model']
    out_tbl = pa.Table.from_pandas(df_en[cols_out], preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PARQ, out_tbl.schema, compression='snappy')
    writer.write_table(out_tbl)
    total_scored += len(df_en)
    elapsed = time.time() - t0
    rate = total_scored / max(elapsed, 1)
    print(f'rg {rg+1}/{pf.num_row_groups}: scored +{len(df_en):,} '
          f'(cum {total_scored:,}/{total_rows:,}, {elapsed:.0f}s, {rate:.0f} titles/s)')

if writer is not None:
    writer.close()
print(f'\nFINAL: {total_scored:,} scored out of {total_rows:,} candidates '
      f'({total_scored/total_rows*100:.1f}%)')
print(f'output: {OUTPUT_PARQ}')

## 6. (선택) finbert_z 정규화 (배포 후 VM 에서 할 수도 있음)

In [ ]:
# 전역 z-score. 월별/업계별 z 는 VM 에서 v10 merge 시 재계산 권장.
import pyarrow.parquet as pq
df = pq.read_table(OUTPUT_PARQ).to_pandas()
mu = df['finbert_score'].mean()
sd = df['finbert_score'].std()
df['finbert_z_global'] = (df['finbert_score'] - mu) / max(sd, 1e-6)
pq.write_table(pa.Table.from_pandas(df, preserve_index=False),
               OUTPUT_PARQ.replace('.parquet', '_with_z.parquet'),
               compression='snappy')
print(df[['finbert_score','finbert_z_global']].describe())

## 7. 결과 다운로드

생성된 `finbert_v2.parquet` 를 VM `/sessions/fervent-tender-clarke/mnt/nabi_hyoghaw/data/processed/` 로 가져와서 `merge_v9_finbert_v10.py` 실행.